# Joint Training: FRCRN + XLSR + AD Classifier

Pipeline: Raw Pitt Audio -> FRCRN Denoise -> XLSR-53 Feature Extraction -> AD Classification

Loss: L_total = alpha * L1_denoise + beta * CE_classify

In [1]:
import sys
from pathlib import Path
import torch
import numpy as np

# Add joint_train and train to path
JOINT_TRAIN_DIR = Path.cwd() if Path.cwd().name == "joint_train" else Path(__file__).parent
sys.path.insert(0, str(JOINT_TRAIN_DIR))
sys.path.insert(0, str(JOINT_TRAIN_DIR.parent / "train"))  # for visualization, data_split

from joint_config import (
    PROJECT_ROOT, RANDOM_SEEDS, RANDOM_SEED, TRAIN_SET_RATIO,
    SAMPLING_RATE, JOINT_BATCH_SIZE, NUM_WORKERS,
)
from joint_dataset import create_joint_dataloaders
from joint_train import train
from data_split import create_split
from visualization import plot_training_curves

## Configuration

In [2]:
# Paths
RAW_AUDIO_DIR = PROJECT_ROOT / "data/raw/Pitt"
CLEAN_AUDIO_DIR = PROJECT_ROOT / "data/denoised/Pitt-MossFormer"  # pseudo clean target
MODEL_OUTPUT_DIR = PROJECT_ROOT / "models/joint_seed_42"

# FRCRN pretrained weights (download from ModelScope if needed)
FRCRN_PRETRAINED_PATH = PROJECT_ROOT / "models/speech_frcrn_ans_cirm_16k/pytorch_model.bin"

print(f"Raw audio:   {RAW_AUDIO_DIR}")
print(f"Clean audio: {CLEAN_AUDIO_DIR}")
print(f"Output:      {MODEL_OUTPUT_DIR}")
print(f"FRCRN ckpt:  {FRCRN_PRETRAINED_PATH}")

Raw audio:   /root/autodl-tmp/Few-Shot_is_all_you_need/ad_detection/data/raw/Pitt
Clean audio: /root/autodl-tmp/Few-Shot_is_all_you_need/ad_detection/data/denoised/Pitt-MossFormer
Output:      /root/autodl-tmp/Few-Shot_is_all_you_need/ad_detection/models/joint_seed_42
FRCRN ckpt:  /root/autodl-tmp/Few-Shot_is_all_you_need/ad_detection/models/speech_frcrn_ans_cirm_16k/pytorch_model.bin


In [3]:
# Device
if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')
print(f"Using {device}")

Using cuda


## Step 1: Train/Validation Split

In [4]:
# Reuse existing Pitt train/val split (same partition as frozen-XLSR experiments)
TRAIN_CSV, VAL_CSV = create_split("Pitt")
print(f"Train CSV: {TRAIN_CSV}")
print(f"Val CSV:   {VAL_CSV}")

Train CSV: /root/autodl-tmp/Few-Shot_is_all_you_need/ad_detection/data/processed/Pitt-xlsr-train.csv
Val CSV:   /root/autodl-tmp/Few-Shot_is_all_you_need/ad_detection/data/processed/Pitt-xlsr-val.csv


## Step 2: Create Data Loaders

In [5]:
train_loader, val_loader = create_joint_dataloaders(
    train_csv=TRAIN_CSV,
    val_csv=VAL_CSV,
    raw_audio_dir=RAW_AUDIO_DIR,
    clean_audio_dir=CLEAN_AUDIO_DIR,
)

JointDataset: 440 samples (Control: 193, Dementia: 247)
JointDataset: 111 samples (Control: 49, Dementia: 62)


In [6]:
# Class weights (handle imbalanced dataset)
num_control = sum(1 for _, l in train_loader.dataset.samples if l == 0)
num_dementia = sum(1 for _, l in train_loader.dataset.samples if l == 1)
total = num_control + num_dementia

class_weight_control = total / (2 * num_control)
class_weight_dementia = total / (2 * num_dementia)

print(f"Control: {num_control}, Dementia: {num_dementia}")
print(f"Class weights: Control={class_weight_control:.4f}, Dementia={class_weight_dementia:.4f}")

Control: 193, Dementia: 247
Class weights: Control=1.1399, Dementia=0.8907


## Step 3: Joint Training (Seed 42)

In [7]:
frcrn_path = str(FRCRN_PRETRAINED_PATH) if FRCRN_PRETRAINED_PATH.exists() else None
if frcrn_path is None:
    print("WARNING: FRCRN pretrained weights not found, training from scratch")

seed, best_metrics, history = train(
    seed=42,
    train_loader=train_loader,
    val_loader=val_loader,
    output_dir=MODEL_OUTPUT_DIR,
    device=device,
    frcrn_pretrained_path=frcrn_path,
    class_weight_control=class_weight_control,
    class_weight_dementia=class_weight_dementia,
)

FRCRN: 13,963,604 total params, 6,570,922 trainable (unet2 only)


/root/autodl-tmp/envs/madress/lib/python3.9/site-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


XLSR: 317,390,592 total params, 37,790,720 trainable (last 3 layers), checkpoint=True


/root/autodl-tmp/Few-Shot_is_all_you_need/ad_detection/joint_train/joint_train.py:229: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)
Epoch 1 - Training:   0%|          | 0/220 [00:00<?, ?it/s]/root/autodl-tmp/Few-Shot_is_all_you_need/ad_detection/joint_train/joint_train.py:81: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=USE_AMP):


OutOfMemoryError: CUDA out of memory. Tried to allocate 1.07 GiB. GPU 0 has a total capacity of 31.36 GiB of which 109.06 MiB is free. Including non-PyTorch memory, this process has 31.24 GiB memory in use. Of the allocated memory 26.64 GiB is allocated by PyTorch, and 3.99 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
plot_training_curves(
    epochs=history['epochs'],
    train_loss=history['train_losses'],
    val_loss=history['val_losses'],
    train_acc=history['train_accs'],
    val_acc=history['val_accs'],
    title_prefix='Joint Training - Seed 42'
)

## Step 4: Results

In [ ]:
print(f"Best Validation Results (Seed {seed}):")
print(f"  Accuracy:     {best_metrics['accuracy'] * 100:.2f}%")
print(f"  F1 Score:     {best_metrics['f1_score']:.4f}")
print(f"  Control Acc:  {best_metrics['control_acc'] * 100:.2f}%")
print(f"  Dementia Acc: {best_metrics['dementia_acc'] * 100:.2f}%")
print(f"  Val Loss:     {best_metrics['loss']:.4f}")
print(f"  Denoise Loss: {best_metrics['denoise_loss']:.4f}")
print(f"  Classify Loss:{best_metrics['classify_loss']:.4f}")

print(f"\nModels saved to: {MODEL_OUTPUT_DIR}")
print(f"  FRCRN:    frcrn_best.pth")
print(f"  XLSR:     xlsr_best.pth")
print(f"  AD Model: ad_model_best.pth")
print(f"  Meta:     meta.pth")

In [ ]:
torch.cuda.empty_cache()